In [40]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, f1_score, classification_report
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, TensorDataset
from statsmodels.miscmodels.ordinal_model import OrderedModel
import joblib

In [41]:
df = pd.read_csv('../data/interim/olist_ml_processed_dataset.csv')

X_raw = df.drop(columns=['actual_delivery_days', 'delivery_status', 'delivery_label']).values
y_raw = df['delivery_label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# Features go to float32; labels go to long (integers) for CrossEntropyLoss
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# --- Step 4: Create DataLoaders ---
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

BATCH_SIZE = 256  # Adjust this depending on your preferences

train_loader = DataLoader(
    dataset=train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True          # Shuffle training data to help generalization
)

test_loader = DataLoader(
    dataset=test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False          # No need to shuffle test/evaluation data
)

print(f"DataLoaders successfully created!")
print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")

DataLoaders successfully created!
Train batches: 271 | Test batches: 68


In [42]:
model = OrderedModel(y_train, X_train, distr='logit')

print("Fitting the Ordinal Logistic Regression model...")

res = model.fit(maxiter=100, method='bfgs', disp=True)

Fitting the Ordinal Logistic Regression model...
Optimization terminated successfully.
         Current function value: 1.539979
         Iterations: 24
         Function evaluations: 25
         Gradient evaluations: 25


In [ ]:
predicted_probs = res.model.predict(res.params, exog=X_test)
predicted_classes = np.argmax(predicted_probs, axis=1)

# Calculate performance metrics
accuracy = accuracy_score(y_test, predicted_classes)
precision = precision_score(y_test, predicted_classes, average='weighted', zero_division=0)
f1 = f1_score(y_test, predicted_classes, average='weighted', zero_division=0)

# Negative Log-Likelihood acts as the training loss
training_loss = res.llf * -1  

print("\n" + "="*55)
print("         OLIST 6-CLASS MODEL PERFORMANCE REPORT         ")
print("="*55)
print(f"Overall Accuracy:   {accuracy * 100:.2f}%")
print(f"Weighted Precision:  {precision * 100:.2f}%")
print(f"Weighted F1-Score:   {f1 * 100:.2f}%")
print("-"*55)
print(f"Final Loss (NLL):    {training_loss:.4f}")
print(f"AIC Score:           {res.aic:.4f}")
print("="*55)

print("\nDetailed Performance Breakdown Across All 6 Classes:")
print(classification_report(y_test, predicted_classes, zero_division=0))


         OLIST 6-CLASS MODEL PERFORMANCE REPORT         
Overall Accuracy:   33.01%
Weighted Precision:  32.22%
Weighted F1-Score:   30.40%
-------------------------------------------------------
Final Loss (NLL):    106601.9765
AIC Score:           213227.9531

Detailed Performance Breakdown Across All 6 Classes:
              precision    recall  f1-score   support

           0       0.28      0.08      0.12      1805
           1       0.34      0.43      0.38      3503
           2       0.31      0.25      0.27      3920
           3       0.34      0.58      0.43      4410
           4       0.27      0.14      0.19      2449
           5       0.42      0.17      0.24      1219

    accuracy                           0.33     17306
   macro avg       0.33      0.27      0.27     17306
weighted avg       0.32      0.33      0.30     17306



In [44]:
model_save_path = '../data/model/ordinal_logistic_regression.pth'
joblib.dump(res, model_save_path)
print(f"\nTrained statsmodels OrderedModel binary successfully saved to '{model_save_path}'")


Trained statsmodels OrderedModel binary successfully saved to '../data/model/ordinal_logistic_regression.pth'
